# Análisis de Víctimas en Carpetas de Investigación FGJ — CDMX

**Objetivo:** Implementar un modelo de clasificación binaria (Logit) para distinguir  
"ROBO A TRANSEUNTE EN VÍA PÚBLICA CON Y SIN VIOLENCIA" vs. "DELITO DE BAJO IMPACTO"  
usando datos de la Fiscalía General de Justicia de la Ciudad de México.

**Fuente:** https://datos.cdmx.gob.mx/dataset/victimas-en-carpetas-de-investigacion-fgj

### ¿Qué nos dicen los datos?

Se trata de datos que contienen la información actualizada de las víctimas de los delitos en las carpetas de investigación de la Fiscalía General de Justicia (FGJ) de la Ciudad de México a partir de enero de 2019.

Para una correcta interpretación de la información, la CDMX hizó las siguientes aclaraciones:

* El campo "fecha_hecho" representa la fecha en la que ocurrió el hecho.
* El campo "fecha_inicio" corresponde a la fecha de la apertura de la carpeta de investigación.
* En esta base se señala el sexo de la víctima, así como la fecha en que ocurrieron los hechos denunciados.
* Es importante destacar que, aunque en algunas ocasiones la víctima es la persona denunciante, en otras, denunciante y víctima son personas diferentes (por ejemplo, casos en los que menores de edad son víctimas).
* Es posible que en una misma denuncia se incluya a una o más víctimas.
* Existen otras aclaraciones en: https://datos.cdmx.gob.mx/dataset/victimas-en-carpetas-de-investigacion-fgj

**Los datos de esta tabla fueron actualizados por la FGJ el 29 de julio de 2020.**

Los datos antes de la actualización del 29 de julio de 2020 los puedes consultar en esta liga: Victimas antes de la actualización de julio de 2020 [https://archivo.datos.cdmx.gob.mx/fiscalia-general-de-justicia/victimas-en-carpetas-de-investigacion-fgj/victimas_ss_junio2020.csv]

## 1. Bibliotecas y configuración

In [ ]:
#
import zipfile
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                             roc_curve, roc_auc_score)
from sklearn.utils.class_weight import compute_class_weight

pd.set_option('display.max_columns', 100)
plt.style.use('ggplot')

# Solo silenciamos convergence warnings específicos, no todo
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings('ignore', category=ConvergenceWarning)

## 2. Funciones auxiliares

In [ ]:
# --- Categorías a eliminar de las series de tiempo (se explica más adelante) ---
CATS_DROP = [
    'HECHO NO DELICTIVO', 'SECUESTRO', 'HOMICIDIO DOLOSO', 'VIOLACIÓN',
    'LESIONES DOLOSAS POR DISPARO DE ARMA DE FUEGO',
    'ROBO A CASA HABITACIÓN CON VIOLENCIA',
    'ROBO A NEGOCIO CON VIOLENCIA',
    'ROBO A REPARTIDOR CON Y SIN VIOLENCIA',
]

#
def drop_categorias(df, extra=None):
    """Elimina columnas de categorías poco frecuentes/no relevantes."""
    cols = CATS_DROP + (extra or [])
    return df.drop(columns=[c for c in cols if c in df.columns])

#
def plot_series(df, title, figsize=(15, 6)):
    """Gráfico de líneas de una tabla pivotada con columna 'Periodo'."""
    plt.figure(figsize=figsize)
    cols = [c for c in df.columns if c != 'Periodo']
    colors = cm.rainbow(np.linspace(0, 1, len(cols)))
    for i, col in enumerate(cols):
        plt.plot(df['Periodo'], df[col], color=colors[i], linewidth=1, alpha=0.9, label=col)
    plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
    plt.title(title, loc='left', fontsize=11, color='darkblue')
    plt.xlabel('Periodo')
    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.show()

#
def build_time_series(df, periodo_col, cat_col, count_col, min_periodo='2019-01'):
    """Agrupa por periodo y categoría, devuelve tabla pivotada."""
    df = df.copy()
    ts = (df[[periodo_col, cat_col, count_col]]
          .groupby([periodo_col, cat_col])
          .count()
          .reset_index()
          .rename(columns={cat_col: 'Categoria', count_col: 'Numero'}))
    ts = ts[ts[periodo_col] >= min_periodo].reset_index(drop=True)
    ts = pd.pivot_table(ts, values='Numero', index=[periodo_col],
                        columns=['Categoria'], aggfunc='sum').reset_index()
    ts.columns.name = None
    return ts

#
DIAS = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
HORAS = [str(h) for h in range(24)]

#
def encode_features(df):
    """One-hot encoding robusto con categorías fijas para días y horas."""
    df = df.copy()
    # Sexo
    df['Femenino'] = (df['sexo'] == 'Femenino').astype(float)
    df['Masculino'] = (df['sexo'] == 'Masculino').astype(float)
    # Personas
    df['Victima'] = (df['personas'] == 'Victima').astype(float)
    df['Victima_y_Otra'] = (df['personas'] == 'Victima_y_Otra').astype(float)
    # Días — categorías fijas
    dia_cat = pd.Categorical(df['fecha_hecho_dia'], categories=DIAS)
    dia_dummies = pd.get_dummies(dia_cat, dtype=float)
    dia_dummies.index = df.index  # Alinear índice para evitar NaN en concat
    # Horas — categorías fijas (0-23)
    hora_cat = pd.Categorical(df['hora_hecho_hora'].astype(str), categories=HORAS)
    hora_dummies = pd.get_dummies(hora_cat, dtype=float)
    hora_dummies.index = df.index  # Alinear índice para evitar NaN en concat
    df = pd.concat([df, dia_dummies, hora_dummies], axis=1)
    return df

# Features para el modelo (referencia de domingo y hora 0 omitidas)
FEATURE_COLS = (['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday']
                + [str(h) for h in range(1, 24)]
                + ['Femenino', 'Victima', 'Dias', 'edad', 'latitud', 'longitud'])


def preparar_dataset(df):
    """Filtra categorías de interés, crea target, aplica encoding y limpia NaN.
    Combina preparar_target + encode_features + limpieza en un solo paso
    para evitar problemas de orden de ejecución.
    """
    # Filtrar categorías
    df = df[(df['categoria_delito'] == 'DELITO DE BAJO IMPACTO') |
            (df['categoria_delito'] == 'ROBO A TRANSEUNTE EN VÍA PÚBLICA CON Y SIN VIOLENCIA')].copy()
    # Target binario
    df['Delito'] = (df['categoria_delito'] == 'ROBO A TRANSEUNTE EN VÍA PÚBLICA CON Y SIN VIOLENCIA').astype(int)
    # Encoding
    df = encode_features(df)
    # Limpiar NaN en features
    mask = df[FEATURE_COLS].notnull().all(axis=1) & np.isfinite(df[FEATURE_COLS]).all(axis=1)
    n_drop = (~mask).sum()
    if n_drop > 0:
        print(f'  Filas con NaN/Inf eliminadas: {n_drop}')
    df = df[mask].copy()
    return df


def entrenar_y_evaluar(X_train, X_test, y_train, y_test, nombre=''):
    """Entrena LogisticRegression, evalúa y muestra resultados completos."""
    logreg = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
    logreg.fit(X_train, y_train)

    y_pred = logreg.predict(X_test)
    y_prob = logreg.predict_proba(X_test)[:, 1]

    # Pesos del balanceo
    classes = np.array([0, 1])
    weights = compute_class_weight('balanced', classes=classes, y=y_train)
    print(f'Balanceo — Clase 0: peso={weights[0]:.4f}, n={(y_train==0).sum()}  |  '
          f'Clase 1: peso={weights[1]:.4f}, n={(y_train==1).sum()}  |  '
          f'Ratio: {weights[1]/weights[0]:.2f}x')
    print()

    # Métricas
    print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
    auc_val = roc_auc_score(y_test, y_prob)
    print(f'AUC-ROC:  {auc_val:.4f}')
    print()
    print(classification_report(y_test, y_pred,
                                target_names=['Bajo impacto (0)', 'Robo transeúnte (1)']))

    # Matriz de confusión
    cm = confusion_matrix(y_test, y_pred)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='YlGnBu',
                xticklabels=['Pred: 0', 'Pred: 1'],
                yticklabels=['Real: 0', 'Real: 1'], ax=axes[0])
    axes[0].set_title(f'Matriz de confusión{" — " + nombre if nombre else ""}')

    # Métricas dinámicas
    TN, FP, FN, TP = cm.ravel()
    print(f'Sensibilidad (Recall):  {TP/(TP+FN):.4f}')
    print(f'Especificidad:          {TN/(TN+FP):.4f}')
    print(f'Precisión (Pos):        {TP/(TP+FP):.4f}')
    print(f'Valor pred. negativo:   {TN/(TN+FN):.4f}')

    # Curva ROC
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    axes[1].plot(fpr, tpr, label=f'Logistic Regression (AUC = {auc_val:.3f})')
    axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Random (AUC = 0.500)')
    axes[1].set_xlim([0, 1.01])
    axes[1].set_ylim([0, 1.01])
    axes[1].set_xlabel('Tasa de falsos positivos (1 - Especificidad)')
    axes[1].set_ylabel('Tasa de verdaderos positivos (Sensibilidad)')
    axes[1].set_title(f'Curva ROC{" — " + nombre if nombre else ""}')
    axes[1].legend()
    axes[1].grid(True)
    plt.tight_layout()
    plt.show()

    return logreg, y_pred, y_prob

## 3. Carga de datos

**Nota:** Los CSV se cargan desde archivos ZIP locales porque el servidor de datos de CDMX
(`archivo.datos.cdmx.gob.mx`) rechaza conexiones desde IPs de datacenters (Google Colab, etc.).

In [ ]:
# Consulta a una dirección Web que aloja un archivo CSV (DATOS ACTUALIZADOS)
#Datos = pd.read_csv('https://archivo.datos.cdmx.gob.mx/FGJ/victimas/victimasFGJ_acumulado_2024_09.csv')

# --- Datos actualizados ---
with zipfile.ZipFile('victimasFGJ_acumulado_2024_09.csv.zip', 'r') as z:
    with z.open('victimasFGJ_acumulado_2024_09.csv') as f:
        Datos = pd.read_csv(f)

print(f'Datos actualizados: {Datos.shape}')
Datos.tail()

In [ ]:
# Consulta a una dirección Web que aloja un archivo CSV (DATOS NO ACTUALIZADOS)
#Datos_OLD = pd.read_csv('https://archivo.datos.cdmx.gob.mx/fiscalia-general-de-justicia/victimas-en-carpetas-de-investigacion-fgj/victimas_ss_junio2020.csv')

# --- Datos NO actualizados (pre-julio 2020) ---
with zipfile.ZipFile('victimas_ss_junio2020.csv.zip', 'r') as z:
    with z.open('victimas_ss_junio2020.csv') as f:
        Datos_OLD = pd.read_csv(f)

print(f'Datos pre-actualización: {Datos_OLD.shape}')
Datos_OLD.tail()

## 4. Exploración: series de tiempo por categoría de delito

In [ ]:
# Datos actualizados — Extracción de fecha
Datos['Periodo'] = Datos['fecha_hecho'].str[:7]

# Usamos función 'build_time_series'
ts = build_time_series(Datos, 'Periodo', 'categoria_delito', 'anio_inicio')

# Eliminamos algunas categorías de delitos que pueden requerir de más conocimiento del dominio:
ts = drop_categorias(ts)
ts

In [ ]:
# Gráficamos
plot_series(ts, 'Víctimas por categoría — Datos actualizados')

In [ ]:
# Sin "DELITO DE BAJO IMPACTO" para ver mejor el resto
plot_series(drop_categorias(ts, extra=['DELITO DE BAJO IMPACTO']),
            'Víctimas por categoría (sin bajo impacto) — Datos actualizados')

In [ ]:
# Datos NO actualizados — Extracción de fecha
Datos_OLD['Periodo'] = Datos_OLD['FechaHecho'].str[6:10] + '-' + Datos_OLD['FechaHecho'].str[3:5]

# Usamos función 'build_time_series'
ts_old = build_time_series(Datos_OLD, 'Periodo', 'Categoria', 'idCarpeta')

# Eliminamos algunas categorías de delitos
ts_old = drop_categorias(ts_old)
ts_old

In [ ]:
# Graficando
plot_series(ts_old, 'Víctimas por categoría — Datos NO actualizados')

In [ ]:
# Sin "DELITO DE BAJO IMPACTO" para ver mejor el resto
plot_series(drop_categorias(ts_old, extra=['DELITO DE BAJO IMPACTO']),
            'Víctimas por categoría (sin bajo impacto) — Datos NO actualizados')

## 5. Limpieza de datos — Datos actualizados

Pasos:
1. Filtrar desde 2019-01
2. Eliminar personas morales
3. Reclasificar calidad jurídica → `personas`
4. Calcular días entre hecho y denuncia
5. Filtrar edad ∈ [0, 100] y días ∈ [0, 365]
6. Filtrar coordenadas nulas

In [ ]:
#
Datos_ML = Datos[Datos['Periodo'] >= '2019-01'].copy()

Datos_ML = Datos_ML[['Periodo', 'fecha_inicio', 'hora_inicio', 'fecha_hecho', 'hora_hecho',
                      'categoria_delito', 'sexo', 'edad', 'tipo_persona',
                      'calidad_juridica', 'latitud', 'longitud']]

print(f'Registros iniciales: {len(Datos_ML):,}')

In [ ]:
# 1. Solo personas físicas
print(Datos_ML['tipo_persona'].value_counts(normalize=True))

Datos_ML = Datos_ML[Datos_ML['tipo_persona'] != 'MORAL']
print(f'Sin personas morales: {len(Datos_ML):,}')

In [ ]:
# 2. Reclasificar calidad jurídica
mapa_victima_otra = ['VICTIMA Y DENUNCIANTE', 'OFENDIDO Y DENUNCIANTE', 'DENUNCIANTE Y VICTIMA']
mapa_victima = ['VICTIMA', 'OFENDIDO', 'AGRAVIADO', 'LESIONADO', 'CADAVER']

# Cómo están los datos:
print(Datos_ML['calidad_juridica'].value_counts(normalize=True))

Datos_ML['personas'] = np.where(
    Datos_ML['calidad_juridica'].isin(mapa_victima_otra), 'Victima_y_Otra',
    np.where(Datos_ML['calidad_juridica'].isin(mapa_victima), 'Victima', 'NA'))

Datos_ML = Datos_ML[Datos_ML['personas'] != 'NA']

# Cómo los vamos a usar:
print(Datos_ML['personas'].value_counts(normalize=True))
print(f'Con calidad jurídica válida: {len(Datos_ML):,}')

In [ ]:
# 3. Días entre hecho y denuncia
Datos_ML['Dias'] = (
    (pd.to_datetime(Datos_ML['fecha_inicio'] + '-' + Datos_ML['hora_inicio'], format='%Y-%m-%d-%H:%M:%S')
     - pd.to_datetime(Datos_ML['fecha_hecho'] + '-' + Datos_ML['hora_hecho'], format='%Y-%m-%d-%H:%M:%S'))
    .dt.total_seconds() / 86400
)

Datos_ML = Datos_ML[(Datos_ML['Dias'] >= 0) & (Datos_ML['Dias'] <= 365)]
print(f'Días ∈ [0, 365]: {len(Datos_ML):,}')

In [ ]:
# 4. Edad válida
Datos_ML = Datos_ML[(Datos_ML['edad'] >= 0) & (Datos_ML['edad'] <= 100)]
print(f'Edad ∈ [0, 100]: {len(Datos_ML):,}')

In [ ]:
# 5. Coordenadas completas (FIX: & en vez de |)
Datos_ML = Datos_ML[Datos_ML['latitud'].notnull() & Datos_ML['longitud'].notnull()]
# Filtrar coordenadas dentro de CDMX (bounding box aproximado)
Datos_ML = Datos_ML[
    (Datos_ML['latitud'] >= 19.0) & (Datos_ML['latitud'] <= 19.8) &
    (Datos_ML['longitud'] >= -99.4) & (Datos_ML['longitud'] <= -98.9)
]
print(f'Con coordenadas y Dentro de CDMX: {len(Datos_ML):,}')

In [ ]:
# 6. Features temporales
Datos_ML['fecha_hecho_dia'] = pd.to_datetime(Datos_ML['fecha_hecho']).dt.day_name()
Datos_ML['hora_hecho_hora'] = pd.to_datetime(Datos_ML['hora_hecho']).dt.hour

print(f'\nRegistros finales: {len(Datos_ML):,}  ({100*len(Datos_ML)/len(Datos):.1f}% del total)')

## 5b. Limpieza de datos — Datos NO actualizados (OLD)

Aplicamos la misma limpieza a los datos pre-julio 2020, adaptando los nombres de columna.

In [ ]:
# Los datos OLD tienen columnas con nombres diferentes, primero exploramos
print('Columnas Datos_OLD:', Datos_OLD.columns.tolist())
print()
print(Datos_OLD.head(2))

In [ ]:
# Renombramos columnas para homologar con los datos actualizados
# Ajusta estos mapeos según lo que veas en la celda anterior
Datos_OLD_ML = Datos_OLD[Datos_OLD['Periodo'] >= '2019-01'].copy()

# Mapeo de columnas OLD → nuevas (ajustar si los nombres difieren)
col_map = {
    'FechaHecho': 'fecha_hecho',
    'FechaInicio': 'fecha_inicio',
    'HoraHecho': 'hora_hecho',
    'HoraInicio': 'hora_inicio',
    'Categoria': 'categoria_delito',
    'Sexo': 'sexo',
    'Edad': 'edad',
    'TipoPersona': 'tipo_persona',
    'CalidadJuridica': 'calidad_juridica',
    'Latitud': 'latitud',
    'Longitud': 'longitud',
}

# Renombrar solo las columnas que existan
col_map_valid = {k: v for k, v in col_map.items() if k in Datos_OLD_ML.columns}
Datos_OLD_ML = Datos_OLD_ML.rename(columns=col_map_valid)

# Verificar qué columnas tenemos
print(f'Registros OLD iniciales: {len(Datos_OLD_ML):,}')
print('Columnas disponibles:', Datos_OLD_ML.columns.tolist())

In [ ]:
# Convertir formatos de fecha si es necesario
# Los datos OLD pueden tener formato DD/MM/YYYY en vez de YYYY-MM-DD
# Detectamos automáticamente:
sample_fecha = Datos_OLD_ML['fecha_hecho'].dropna().iloc[0]
print(f'Formato de fecha ejemplo: "{sample_fecha}"')

if '/' in str(sample_fecha):
    # Formato DD/MM/YYYY → convertir a YYYY-MM-DD
    Datos_OLD_ML['fecha_hecho'] = pd.to_datetime(Datos_OLD_ML['fecha_hecho'],
                                                   format='%d/%m/%Y', errors='coerce').dt.strftime('%Y-%m-%d')
    if 'fecha_inicio' in Datos_OLD_ML.columns:
        Datos_OLD_ML['fecha_inicio'] = pd.to_datetime(Datos_OLD_ML['fecha_inicio'],
                                                       format='%d/%m/%Y', errors='coerce').dt.strftime('%Y-%m-%d')
    print('Fechas convertidas de DD/MM/YYYY a YYYY-MM-DD')
else:
    print('Fechas ya están en formato esperado')

In [ ]:
# Aplicamos la misma limpieza que a los datos actualizados
# Seleccionar columnas necesarias (solo las que existan)
cols_needed = ['Periodo', 'fecha_inicio', 'hora_inicio', 'fecha_hecho', 'hora_hecho',
               'categoria_delito', 'sexo', 'edad', 'tipo_persona',
               'calidad_juridica', 'latitud', 'longitud']

cols_available = [c for c in cols_needed if c in Datos_OLD_ML.columns]
cols_missing = [c for c in cols_needed if c not in Datos_OLD_ML.columns]
if cols_missing:
    print(f'⚠️ Columnas faltantes en OLD: {cols_missing}')
    print('El modelo OLD usará solo las columnas disponibles.')

Datos_OLD_ML = Datos_OLD_ML[cols_available]
print(f'Columnas seleccionadas: {len(cols_available)}')

In [ ]:
# 1. Solo personas físicas (si la columna existe)
if 'tipo_persona' in Datos_OLD_ML.columns:
    print(Datos_OLD_ML['tipo_persona'].value_counts(normalize=True))
    Datos_OLD_ML = Datos_OLD_ML[Datos_OLD_ML['tipo_persona'] != 'MORAL']
print(f'Sin personas morales: {len(Datos_OLD_ML):,}')

In [ ]:
# 2. Reclasificar calidad jurídica
if 'calidad_juridica' in Datos_OLD_ML.columns:
    print(Datos_OLD_ML['calidad_juridica'].value_counts(normalize=True))
    Datos_OLD_ML['personas'] = np.where(
        Datos_OLD_ML['calidad_juridica'].isin(mapa_victima_otra), 'Victima_y_Otra',
        np.where(Datos_OLD_ML['calidad_juridica'].isin(mapa_victima), 'Victima', 'NA'))
    Datos_OLD_ML = Datos_OLD_ML[Datos_OLD_ML['personas'] != 'NA']
else:
    # Si no hay calidad_juridica, asumimos todos son Victima
    Datos_OLD_ML['personas'] = 'Victima'
    print('⚠️ Sin columna calidad_juridica, asumiendo todos Victima')

print(f'Con calidad jurídica válida: {len(Datos_OLD_ML):,}')

In [ ]:
# 3. Días entre hecho y denuncia
if 'fecha_inicio' in Datos_OLD_ML.columns and 'hora_inicio' in Datos_OLD_ML.columns:
    Datos_OLD_ML['Dias'] = (
        (pd.to_datetime(Datos_OLD_ML['fecha_inicio'] + '-' + Datos_OLD_ML['hora_inicio'],
                        format='%Y-%m-%d-%H:%M:%S', errors='coerce')
         - pd.to_datetime(Datos_OLD_ML['fecha_hecho'] + '-' + Datos_OLD_ML['hora_hecho'],
                          format='%Y-%m-%d-%H:%M:%S', errors='coerce'))
        .dt.total_seconds() / 86400
    )
    Datos_OLD_ML = Datos_OLD_ML[(Datos_OLD_ML['Dias'] >= 0) & (Datos_OLD_ML['Dias'] <= 365)]
else:
    Datos_OLD_ML['Dias'] = 0
    print('⚠️ Sin fechas de inicio, Dias=0 por defecto')

print(f'Días ∈ [0, 365]: {len(Datos_OLD_ML):,}')

In [ ]:
# 4. Edad válida
Datos_OLD_ML = Datos_OLD_ML[(Datos_OLD_ML['edad'] >= 0) & (Datos_OLD_ML['edad'] <= 100)]
print(f'Edad ∈ [0, 100]: {len(Datos_OLD_ML):,}')

In [ ]:
# 5. Coordenadas completas
Datos_OLD_ML = Datos_OLD_ML[Datos_OLD_ML['latitud'].notnull() & Datos_OLD_ML['longitud'].notnull()]
# Filtrar coordenadas dentro de CDMX (bounding box aproximado)
Datos_OLD_ML = Datos_OLD_ML[
    (Datos_OLD_ML['latitud'] >= 19.0) & (Datos_OLD_ML['latitud'] <= 19.8) &
    (Datos_OLD_ML['longitud'] >= -99.4) & (Datos_OLD_ML['longitud'] <= -98.9)
]
print(f'Con coordenadas y Dentro de CDMX: {len(Datos_OLD_ML):,}')

In [ ]:
# 6. Features temporales
Datos_OLD_ML['fecha_hecho_dia'] = pd.to_datetime(Datos_OLD_ML['fecha_hecho']).dt.day_name()
Datos_OLD_ML['hora_hecho_hora'] = pd.to_datetime(Datos_OLD_ML['hora_hecho'], errors='coerce').dt.hour

print(f'\nRegistros OLD finales: {len(Datos_OLD_ML):,}')

## 6. Exploración de datos limpios

In [ ]:
# Datos ML
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Días de la semana
orden_dias = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
sns.countplot(x='fecha_hecho_dia', data=Datos_ML, order=orden_dias,
              palette='Set2', ax=axes[0, 0])
axes[0, 0].set_title('Hechos por día de la semana \n¿Qué día de la semana se registran más delitos?')
axes[0, 0].tick_params(axis='x', rotation=45)

# Hora del día
Datos_ML['hora_hecho_hora'].hist(bins=24, ax=axes[0, 1], color='steelblue', edgecolor='white')
axes[0, 1].set_title('Distribución por hora del día \n¿Qué hora del día se registran más delitos?')

# Edad
Datos_ML['edad'].hist(bins=30, ax=axes[1, 0], color='coral', edgecolor='white')
axes[1, 0].set_title('Distribución de edad \n¿Qué edades se observan en los registros de delitos?')

# Días entre hecho y denuncia
Datos_ML['Dias'].hist(bins=30, ax=axes[1, 1], color='mediumseagreen', edgecolor='white')
axes[1, 1].set_title('Días entre hecho y denuncia \n¿Cuánto tiempo le toma a las personas denunciar?')

plt.tight_layout()
plt.show()

In [ ]:
# Datos OLD
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Días de la semana
orden_dias = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
sns.countplot(x='fecha_hecho_dia', data=Datos_OLD_ML, order=orden_dias,
              palette='Set2', ax=axes[0, 0])
axes[0, 0].set_title('Hechos por día de la semana \n¿Qué día de la semana se registran más delitos?')
axes[0, 0].tick_params(axis='x', rotation=45)

# Hora del día
Datos_OLD_ML['hora_hecho_hora'].hist(bins=24, ax=axes[0, 1], color='steelblue', edgecolor='white')
axes[0, 1].set_title('Distribución por hora del día \n¿Qué hora del día se registran más delitos?')

# Edad
Datos_OLD_ML['edad'].hist(bins=30, ax=axes[1, 0], color='coral', edgecolor='white')
axes[1, 0].set_title('Distribución de edad \n¿Qué edades se observan en los registros de delitos?')

# Días entre hecho y denuncia
Datos_OLD_ML['Dias'].hist(bins=30, ax=axes[1, 1], color='mediumseagreen', edgecolor='white')
axes[1, 1].set_title('Días entre hecho y denuncia \n¿Cuánto tiempo le toma a las personas denunciar?')

plt.tight_layout()
plt.show()

In [ ]:
# Mapa de puntos
Datos_ML.plot(kind='scatter', x='longitud', y='latitud', alpha=0.01, figsize=(8, 8),
              title='Distribución geográfica de hechos')
plt.show()

In [ ]:
# Mapa de puntos
Datos_OLD_ML.plot(kind='scatter', x='longitud', y='latitud', alpha=0.01, figsize=(8, 8),
              title='Distribución geográfica de hechos')
plt.show()

## 7. Preparación de datasets para modelado — Datos actualizados

- **Datos_01** (2019): para entrenar y evaluar el modelo.
- **Datos_02** (2023): para predecir/clasificar con el modelo entrenado (para dejar fuera los años de la pandemia).

**Objetivo**: Construyamos un modelo que aprenda a clasificar delitos como lo hacen las autoridades de la CDMX (para efectos de la clase, solo usaremos 2 categorías y usamos años completos para evitar sesgos por estacionalidad, por ejemplo).

In [ ]:
#
Datos_01 = Datos_ML[(Datos_ML['Periodo'] >= '2019-01') & (Datos_ML['Periodo'] <= '2019-12')].copy()
Datos_02 = Datos_ML[(Datos_ML['Periodo'] >= '2023-01') & (Datos_ML['Periodo'] <= '2023-12')].copy()

print(f'Datos 2019 (train): {Datos_01.shape}')
print(f'Datos 2023 (predict): {Datos_02.shape}')

In [ ]:
# preparar_dataset: filtra categorías + encoding + limpieza NaN en un solo paso
# Esto evita el bug de perder el encoding al re-ejecutar celdas fuera de orden
Datos_01 = preparar_dataset(Datos_01)
Datos_02 = preparar_dataset(Datos_02)

print(f'Datos 2019 listos: {Datos_01.shape}  — Tasa positiva: {Datos_01.Delito.mean():.2%}')
print(f'Datos 2023 listos: {Datos_02.shape}  — Tasa positiva: {Datos_02.Delito.mean():.2%}')

In [ ]:
# Comparación entre train y test
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.countplot(x='Delito', data=Datos_01, ax=axes[0], palette='Set1')
axes[0].set_title('2019 — Distribución de clases')
sns.countplot(x='Delito', data=Datos_02, ax=axes[1], palette='Set1')
axes[1].set_title('2023 — Distribución de clases')
plt.tight_layout()
plt.show()

## 7b. Preparación de datasets para modelado — Datos OLD

Aplicamos el mismo proceso a los datos NO actualizados para comparar el desempeño del modelo.

In [ ]:
# Preparar dataset OLD
Datos_OLD_ready = preparar_dataset(Datos_OLD_ML)

print(f'Datos OLD listos: {Datos_OLD_ready.shape}  — Tasa positiva: {Datos_OLD_ready.Delito.mean():.2%}')

In [ ]:
sns.countplot(x='Delito', data=Datos_OLD_ready, palette='Set1')
plt.title('Datos OLD — Distribución de clases')
plt.show()

## 8. Modelo Logit — Statsmodels (inferencia estadística)

* Plantearemos un modelo de respuesta binaria:

Este tipo de modelos suponene que existe una variable latente que se puede expresar como una ecuación lineal dada por:
$$y^*_i = \mathbf{x}_i \boldsymbol{\beta} + \varepsilon_i$$

Donde $\varepsilon_i$ es una variable aleatoria con función de densidad con media cero y distribución simetrica al rededor de cero. Dado lo anterior, para nosotros sólo es visible que:
\begin{equation*}
    y_i =
    \begin{cases}
        1 & \text{si } y^*_i > 0 \\
        0 & \text{si } y^*_i < 0
    \end{cases}
\end{equation*}

Visto en nuestro caso, podemos pensar que estamos ante algo como:
\begin{equation*}
    y_i =
    \begin{cases}
        \text{ ROBO A TRANSEUNTE EN VÍA PÚBLICA CON Y SIN VIOLENCIA } & \text{si } y^*_i > 0 \\
        \text{ OTRO CASO } & \text{si } y^*_i < 0
    \end{cases}
\end{equation*}

De esta forma tenemos una estrucutura de la probabilidad dada por:
\begin{eqnarray*}
    P(y_i = 1 | \mathbf{x}_i) & = & P(\mathbf{x}_i \boldsymbol{\beta} + \varepsilon_i > 0) = P(\varepsilon_i > - \mathbf{x}_i \boldsymbol{\beta}) = G(\mathbf{x}_i \boldsymbol{\beta}) \\
    P(y_i = 0 | \mathbf{x}_i) & = & P(\mathbf{x}_i \boldsymbol{\beta} + \varepsilon_i < 0) = P(\varepsilon_i < - \mathbf{x}_i \boldsymbol{\beta}) = 1 - G(\mathbf{x}_i \boldsymbol{\beta})
\end{eqnarray*}

Donde $\mathbf{x}_i$ es un vector de dimensión $K \times 1$ que contiene al menos el término constante y $\boldsymbol{\beta}$ es un vector de parámetros a estimar, de forma que asumiremos:
\begin{equation*}
    \mathbf{x}_i \boldsymbol{\beta} = \beta_1 + x_{i2} \beta_2 + \ldots + x_{iK} \beta_K
\end{equation*}

Asumiremos que $G(\cdot)$ es uan función de densidad acumulada de forma que:
\begin{equation*}
    0< G(\mathbf{x}_i \boldsymbol{\beta}) < 1 \text{ , } \forall \mathbf{x}_i \boldsymbol{\beta} \in \mathbb{R}
\end{equation*}

En este caso utilizaremos dos modelos que dependen de la forma funcional de $G(\cdot)$ que está determinada por la distribución de $\varepsilon_i$. De esta forma tendremos dos modelos: Probit y Logit.

-------------------------
Estimamos primero con `statsmodels` para obtener p-values, z-scores y pseudo-R².

In [ ]:
# Usando las variables en FEATURE_COLS
X_sm = sm.add_constant(Datos_01[FEATURE_COLS])
y_sm = Datos_01['Delito']

model_sm = sm.Logit(y_sm, X_sm)
result_sm = model_sm.fit(disp=0)
print(result_sm.summary())

## 9. Modelo Logit — Sklearn (clasificación) — Datos actualizados

Usamos `class_weight='balanced'` para manejar el desbalance de clases automáticamente, en vez de un rebalanceo manual.

**¿Por qué hacer un rebalanceo?**

HINT: mira las gráficas de barras de arriba.

In [ ]:
#
X = Datos_01[FEATURE_COLS]
y = Datos_01['Delito']

# Separación de datasets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

# Entrenar y evaluar
logreg, y_pred, y_prob = entrenar_y_evaluar(X_train, X_test, y_train, y_test, nombre='2019')

### Importancia de features (coeficientes del Logit)

In [ ]:
#
coefs = pd.DataFrame({
    'Feature': FEATURE_COLS,
    'Coeficiente': logreg.coef_[0]
}).sort_values('Coeficiente', key=abs, ascending=True)

plt.figure(figsize=(8, 10))
plt.barh(coefs['Feature'], coefs['Coeficiente'], color='steelblue')
plt.xlabel('Coeficiente')
plt.title('Importancia de features — Logistic Regression (Datos 2019)')
plt.tight_layout()
plt.show()

## 10. Predicción sobre datos 2023

Aplicamos el modelo entrenado con datos 2019 para clasificar los registros de 2023.

In [ ]:
#
X_new = Datos_02[FEATURE_COLS]
Datos_02['Delito_Predict'] = logreg.predict(X_new)
Datos_02['Prob_Robo'] = logreg.predict_proba(X_new)[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(x='Delito', data=Datos_02, ax=axes[0], palette='Set1')
axes[0].set_title('2023 — Clasificación real')
sns.countplot(x='Delito_Predict', data=Datos_02, ax=axes[1], palette='Set2')
axes[1].set_title('2023 — Clasificación predicha')
plt.tight_layout()
plt.show()

print('\nComparación real vs predicho:')
print(Datos_02[['Delito', 'Delito_Predict']].describe())

In [ ]:
# Evaluación formal en datos 2023
cm_new = confusion_matrix(Datos_02['Delito'], Datos_02['Delito_Predict'])
print(classification_report(Datos_02['Delito'], Datos_02['Delito_Predict'],
                            target_names=['Bajo impacto (0)', 'Robo transeúnte (1)']))

auc_new = roc_auc_score(Datos_02['Delito'], Datos_02['Prob_Robo'])
print(f'AUC-ROC en datos 2023: {auc_new:.4f}')

## 11. Modelo Logit — Datos NO actualizados (OLD)

Entrenamos y evaluamos el mismo modelo sobre los datos pre-julio 2020 para comparar
si los patrones delictivos son consistentes entre ambos datasets.

In [ ]:
#
X_old = Datos_OLD_ready[FEATURE_COLS]
y_old = Datos_OLD_ready['Delito']

X_train_old, X_test_old, y_train_old, y_test_old = train_test_split(
    X_old, y_old, test_size=0.20, random_state=42, stratify=y_old)

logreg_old, y_pred_old, y_prob_old = entrenar_y_evaluar(
    X_train_old, X_test_old, y_train_old, y_test_old, nombre='OLD')

In [ ]:
# Feature importance OLD
coefs_old = pd.DataFrame({
    'Feature': FEATURE_COLS,
    'Coeficiente': logreg_old.coef_[0]
}).sort_values('Coeficiente', key=abs, ascending=True)

plt.figure(figsize=(8, 10))
plt.barh(coefs_old['Feature'], coefs_old['Coeficiente'], color='coral')
plt.xlabel('Coeficiente')
plt.title('Importancia de features — Logistic Regression (Datos OLD)')
plt.tight_layout()
plt.show()

### Predicción cruzada: modelo OLD → datos 2023 y modelo 2019 → datos OLD

Esto permite evaluar la transferibilidad temporal de los modelos.

In [ ]:
# Modelo OLD prediciendo datos 2023
print('=== Modelo OLD → Datos 2023 ===')
y_pred_cross1 = logreg_old.predict(Datos_02[FEATURE_COLS])
y_prob_cross1 = logreg_old.predict_proba(Datos_02[FEATURE_COLS])[:, 1]
print(classification_report(Datos_02['Delito'], y_pred_cross1,
                            target_names=['Bajo impacto (0)', 'Robo transeúnte (1)']))
print(f'AUC-ROC: {roc_auc_score(Datos_02["Delito"], y_prob_cross1):.4f}')

In [ ]:
# Modelo 2019 prediciendo datos OLD
print('=== Modelo 2019 → Datos OLD ===')
y_pred_cross2 = logreg.predict(Datos_OLD_ready[FEATURE_COLS])
y_prob_cross2 = logreg.predict_proba(Datos_OLD_ready[FEATURE_COLS])[:, 1]
print(classification_report(Datos_OLD_ready['Delito'], y_pred_cross2,
                            target_names=['Bajo impacto (0)', 'Robo transeúnte (1)']))
print(f'AUC-ROC: {roc_auc_score(Datos_OLD_ready["Delito"], y_prob_cross2):.4f}')

### Comparación de modelos

In [ ]:
# Resumen comparativo
print('╔══════════════════════════════════════════════════════════════╗')
print('║              RESUMEN COMPARATIVO DE MODELOS                 ║')
print('╠══════════════════════════════════════════════════════════════╣')
print(f'║ Modelo 2019 — AUC train/test:  {roc_auc_score(y_test, y_prob):.4f}                       ║')
print(f'║ Modelo 2019 → Datos 2023:      {roc_auc_score(Datos_02["Delito"], Datos_02["Prob_Robo"]):.4f}                       ║')
print(f'║ Modelo 2019 → Datos OLD:       {roc_auc_score(Datos_OLD_ready["Delito"], y_prob_cross2):.4f}                       ║')
print('╠══════════════════════════════════════════════════════════════╣')
print(f'║ Modelo OLD  — AUC train/test:  {roc_auc_score(y_test_old, y_prob_old):.4f}                       ║')
print(f'║ Modelo OLD  → Datos 2023:      {roc_auc_score(Datos_02["Delito"], y_prob_cross1):.4f}                       ║')
print('╚══════════════════════════════════════════════════════════════╝')

## 12. Conclusiones

### Hallazgos principales
1. **Variables más relevantes:** Revisar los gráficos de coeficientes para identificar qué features
   (hora, día, ubicación, sexo, edad) tienen mayor poder predictivo para robo a transeúnte.
2. **Desempeño:** El AUC-ROC en validación cruzada indica qué tan generalizable es el modelo.
3. **Transferibilidad temporal:** Comparar el AUC en datos 2019 vs 2023 vs OLD permite evaluar
   si los patrones delictivos se mantienen estables en el tiempo.
4. **Consistencia entre datasets:** La predicción cruzada (modelo OLD → datos 2023 y viceversa)
   muestra qué tanto se preservan los patrones entre los datos actualizados y no actualizados.

